In [ ]:
# Стандартная библиотека
import gc
import logging
import os
import pickle
import sys
import warnings
from collections import defaultdict
from pathlib import Path
from typing import List, Tuple
import os
import sys
import warnings

import pandas as pd

# Работа с данными
import numpy as np
import pandas as pd

# Визуализация
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from plotly.subplots import make_subplots

# Статистика
from scipy.sparse import csr_matrix
from scipy.stats import gaussian_kde

# ML
import lightgbm as lgb
from implicit.als import AlternatingLeastSquares
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Файлы и хранилища
import boto3
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs
from dotenv import load_dotenv

# Прочее
import phik
from phik import report, resources
from tqdm.auto import tqdm

In [ ]:
# настройки отображения
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
# отключаем научную нотацию для удобаства анализа данных
pd.options.display.float_format = '{:,.0f}'.format

# настройки графиков
%matplotlib inline
%config InlineBackend.figure_format = "retina"

# корень проекта
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("PROJECT_ROOT:", os.path.basename(PROJECT_ROOT))
print("src exists:", os.path.isdir(os.path.join(PROJECT_ROOT, "src")))

def to_relative(path, base):
    try:
        return os.path.relpath(path, base)
    except ValueError:
        return path

from src.utils.config import (
    DATA_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    EVENTS_PATH,
    CATEGORY_TREE_PATH,
    ITEM_PROPERTIES_PATH,
    ARTIFACTS_DIR,
    MODELS_DIR,
    MLFLOW_BASE_DIR,
    MLFLOW_DIR,
    MLFLOW_DB_PATH,
    AIRFLOW_DIR,
    AIRFLOW_DAGS_DIR,
)

# S3
S3_BASE = "s3://s3-student-mle-20250927-31ecef0a74/recsys"
S3_DATA_DIR = f"{S3_BASE}/data"
S3_REC_DIR = f"{S3_BASE}/recommendations"

# проверка путей
paths_to_check = {
    "DATA_DIR": DATA_DIR,
    "RAW_DIR": RAW_DIR,
    "PROCESSED_DIR": PROCESSED_DIR,
    "ARTIFACTS_DIR": ARTIFACTS_DIR,
    "MODELS_DIR": MODELS_DIR,
    "MLFLOW_BASE_DIR": MLFLOW_BASE_DIR,
    "MLFLOW_DIR": MLFLOW_DIR,
    "MLFLOW_DB_PATH": MLFLOW_DB_PATH,
    "AIRFLOW_DIR": AIRFLOW_DIR,
    "AIRFLOW_DAGS_DIR": AIRFLOW_DAGS_DIR,
    "EVENTS_PATH": EVENTS_PATH,
    "CATEGORY_TREE_PATH": CATEGORY_TREE_PATH,
    "ITEM_PROPERTIES_PATH": ITEM_PROPERTIES_PATH,
}

print("\nProject paths:\n")

for name, path in paths_to_check.items():
    rel_path = to_relative(path, PROJECT_ROOT)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"{name:<22} {rel_path:<40} [{status}]")